# Inference & Interactive Generation

Notebook demo for loading the best trained model and generating Shakespeare-style text (`proposal.md` §8).

## §8.1 Notebook-Based Inference

- Load the best model from a local checkpoint or W&B artifact (`sweep-best-v1`)
- **Input**: text prompt (e.g. `"ROMEO:"`) and max generation length
- **Output**: autoregressive continuation

## §8.2 Interactive Script Generator

- Adjust **temperature**, **top-k**, and **top-p** sampling
- Compare outputs for the same prompt under different settings

## Prerequisites

1. `2. Preprocessing.ipynb` → `data/artifacts/char_vocab.json`
2. `5. Model Training.ipynb` → `data/artifacts/model/gpt_best.pt` (or W&B artifact)
3. For interactive UI: `pip install ipywidgets`

In [1]:
from pathlib import Path
import json
import math
import copy
import tempfile

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
from IPython.display import display, Markdown, clear_output

try:
    import wandb
except ImportError:
    wandb = None

try:
    import ipywidgets as widgets
    from ipywidgets import interact, interactive_output
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed — interactive section will be skipped. Run: pip install ipywidgets")

cwd = Path.cwd()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError(f"Could not locate 'data' from cwd: {cwd}")

DATA_DIR = PROJECT_ROOT / "data"
ARTIFACTS_DIR = DATA_DIR / "artifacts"
CHECKPOINT_DIR = ARTIFACTS_DIR / "model"

LOCAL_CKPT = CHECKPOINT_DIR / "gpt_best.pt"
VOCAB_PATH = ARTIFACTS_DIR / "char_vocab.json"
EVAL_REPORT_PATH = CHECKPOINT_DIR / "evaluation_report.json"

INFERENCE_CONFIG = {
    "wandb_project": "genre-story-generator",
    "wandb_entity": None,
    "wandb_artifact": "sweep-best-v1",
    "load_source": "local",  # "local" | "wandb"
    "default_max_tokens": 400,
    "seed": 42,
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda


## Load Vocabulary

In [2]:
assert VOCAB_PATH.exists(), f"Missing {VOCAB_PATH}. Run preprocessing notebook first."

with open(VOCAB_PATH, "r", encoding="utf-8") as f:
    vocab_payload = json.load(f)

vocab_size = vocab_payload["vocab_size"]
stoi = vocab_payload["stoi"]
itos = {int(k): v for k, v in vocab_payload["itos"].items()}


def encode(text: str) -> list[int]:
    unknown = [ch for ch in text if ch not in stoi]
    if unknown:
        print(f"Warning: {len(unknown)} character(s) not in vocab (ignored): {set(unknown)}")
    return [stoi[ch] for ch in text if ch in stoi]


def decode(indices) -> str:
    return "".join(itos[int(i)] for i in indices)


print(f"Vocabulary size: {vocab_size}")

Vocabulary size: 65


## Model Architecture

Same decoder-only GPT as notebooks 4–6 (inference only — no training).

In [3]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = d_model // n_heads
        self.qkv_proj = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("mask", None, persistent=False)

    def _causal_mask(self, size: int, device: torch.device):
        if self.mask is None or self.mask.size(0) < size:
            self.mask = torch.tril(torch.ones(size, size, device=device)).view(1, 1, size, size)
        return self.mask[:, :, :size, :size]

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()
        qkv = self.qkv_proj(x)
        q, k, v = qkv.chunk(3, dim=-1)

        def reshape_heads(t):
            return t.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q, k, v = reshape_heads(q), reshape_heads(k), reshape_heads(v)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        mask = self._causal_mask(T, x.device)
        att = att.masked_fill(mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(y))


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff_mult: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff_mult * d_model),
            nn.GELU(),
            nn.Linear(d_ff_mult * d_model, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size: int, config: dict):
        super().__init__()
        d_model = config["d_model"]
        n_heads = config["n_heads"]
        n_layers = config["n_layers"]
        dropout = config["dropout"]
        block_size = config["block_size"]
        d_ff_mult = config.get("d_ff", 4)
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_embedding = nn.Embedding(block_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList(
            [TransformerBlock(d_model, n_heads, d_ff_mult, dropout) for _ in range(n_layers)]
        )
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.size()
        if T > self.block_size:
            raise ValueError(f"Sequence length {T} exceeds block_size {self.block_size}")
        pos = torch.arange(0, T, device=idx.device, dtype=torch.long).unsqueeze(0)
        x = self.drop(self.token_embedding(idx) + self.pos_embedding(pos))
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B * T, -1), targets.view(B * T))
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int = 0,
        top_p: float = 1.0,
    ):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-8)
            if top_k > 0:
                k = min(top_k, logits.size(-1))
                v, _ = torch.topk(logits, k, dim=-1)
                logits = logits.masked_fill(logits < v[:, [-1]], float("-inf"))
            if top_p < 1.0:
                sorted_logits, sorted_idx = torch.sort(logits, descending=True, dim=-1)
                cum_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
                remove = cum_probs - F.softmax(sorted_logits, dim=-1) >= top_p
                sorted_logits[remove] = float("-inf")
                logits = sorted_logits.scatter(1, sorted_idx, sorted_logits)
            probs = F.softmax(logits, dim=-1)
            idx = torch.cat([idx, torch.multinomial(probs, num_samples=1)], dim=1)
        return idx

## Load Best Model

Loads from **local** `gpt_best.pt` by default. Set `INFERENCE_CONFIG["load_source"] = "wandb"` to download the `sweep-best-v1` artifact (proposal §8.1).

In [4]:
def _load_checkpoint_file(ckpt_path: Path):
    ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
    if isinstance(ckpt, dict) and "model" in ckpt and "config" in ckpt:
        return ckpt["model"], copy.deepcopy(ckpt["config"])
    raise ValueError(f"Unexpected checkpoint format at {ckpt_path}")


def load_from_local(path: Path = LOCAL_CKPT):
    assert path.exists(), f"Missing {path}. Run notebook 5 first."
    state, config = _load_checkpoint_file(path)
    return state, config, f"local:{path.name}"


def load_from_wandb_artifact(artifact_name: str = INFERENCE_CONFIG["wandb_artifact"]):
    if wandb is None:
        raise ImportError("wandb required for artifact download. pip install wandb && wandb login")

    api = wandb.Api()
    entity = INFERENCE_CONFIG.get("wandb_entity") or api.default_entity
    project = INFERENCE_CONFIG["wandb_project"]
    artifact = api.artifact(f"{entity}/{project}/{artifact_name}:latest")

    download_dir = Path(tempfile.mkdtemp(prefix="wandb_artifact_"))
    artifact.download(root=str(download_dir))

    ckpt_candidates = list(download_dir.rglob("gpt_best.pt")) + list(download_dir.rglob("*.pt"))
    if not ckpt_candidates:
        raise FileNotFoundError(f"No .pt checkpoint found in artifact {artifact_name}")

    ckpt_path = ckpt_candidates[0]
    state, config = _load_checkpoint_file(ckpt_path)
    return state, config, f"wandb:{artifact_name}"


def build_model(state: dict, config: dict) -> GPTLanguageModel:
    config["block_size"] = config.get("block_size", 128)
    model = GPTLanguageModel(vocab_size=vocab_size, config=config).to(device)
    model.load_state_dict(state)
    model.eval()
    return model


if INFERENCE_CONFIG["load_source"] == "wandb":
    state_dict, model_config, load_label = load_from_wandb_artifact()
else:
    state_dict, model_config, load_label = load_from_local()

model = build_model(state_dict, model_config)
n_params = sum(p.numel() for p in model.parameters())

print(f"Loaded model from {load_label}")
print(
    f"  params={n_params:,} | n_layers={model_config['n_layers']} | "
    f"d_model={model_config['d_model']} | block_size={model_config['block_size']}"
)

Loaded model from local:gpt_best.pt
  params=15,895,040 | n_layers=5 | d_model=512 | block_size=128


## §8.1 Inference API

Core function: **prompt** + **max_new_tokens** → generated text.

In [5]:
def set_seed(seed: int) -> None:
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


@torch.no_grad()
def generate_text(
    prompt: str,
    max_new_tokens: int = INFERENCE_CONFIG["default_max_tokens"],
    temperature: float = 0.8,
    top_k: int = 40,
    top_p: float = 0.9,
    seed: int | None = INFERENCE_CONFIG["seed"],
) -> str:
    """
    Autoregressively continue `prompt` using the loaded GPT.

    Returns the full string (prompt + continuation).
    """
    if seed is not None:
        set_seed(seed)

    ids = encode(prompt)
    if not ids:
        raise ValueError("Prompt produced no valid tokens. Use characters from the training vocabulary.")

    context = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(
        context,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
    )
    return decode(out[0].tolist())


def print_generation(prompt: str, **kwargs) -> str:
    text = generate_text(prompt, **kwargs)
    print(text)
    return text

### Example: `ROMEO:` prompt (proposal §8.1)

In [6]:
demo_prompt = "ROMEO:"
demo_text = print_generation(
    demo_prompt,
    max_new_tokens=300,
    temperature=0.8,
    top_k=40,
    top_p=0.9,
)

ROMEO:
The king shall be my true moving liege,
And see how the good for ladies to my father,
I tell thee thou art a sickle child,
Thou hast the partiest to the hand with thy hand,
And bid them with the high seat in his foe.

MARIANA:
Have you no matter?

ISABELLA:
Then I have been the common brother well 


### Preset character prompts

In [7]:
PRESET_PROMPTS = [
    "ROMEO:",
    "JULIET:",
    "KING:",
    "LADY CAPULET:",
    "First Citizen:",
]

for p in PRESET_PROMPTS:
    print("=" * 72)
    print(p)
    print("-" * 72)
    out = generate_text(p, max_new_tokens=250, temperature=0.8, top_k=40, top_p=0.9, seed=42)
    print(out[:500])
    if len(out) > 500:
        print("...")

ROMEO:
------------------------------------------------------------------------
ROMEO:
The king shall be my true moving liege,
And see how the good for ladies to my father,
I tell thee thou art a sickle child,
Thou hast the partiest to the hand with thy hand,
And bid them with the high seat in his foe.

MARIANA:
Have you no matter?

I
JULIET:
------------------------------------------------------------------------
JULIET:
The king is the news to the duke:
How came is the envy for the walls?

Nurse:
The queen, she makes a dangerous colour.

KING RICHARD II:
Then saw the reason are mercy against the heart:
The worst is now with weak woe, now changing the sea,
Shall be 
KING:
------------------------------------------------------------------------
KING:

GLOUCESTER:
Why, what says he that he hath some heart of him?

KING EDWARD IV:
So shall we the state and be so many for a goodly shoulder.

GLOUCESTER:
Have we met all the shadow of him,
That our consent marriage can no measure of the kin

## Compare Sampling Strategies

Same prompt, different **temperature** / **top-k** / **top-p** (proposal §8.2 — prevents degenerate loops).

In [8]:
COMPARE_PROMPT = "ROMEO:"

SAMPLING_STRATEGIES = [
    {
        "label": "Conservative (temp=0.5)",
        "temperature": 0.5,
        "top_k": 0,
        "top_p": 1.0,
    },
    {
        "label": "Balanced (temp=0.8, top_k=40)",
        "temperature": 0.8,
        "top_k": 40,
        "top_p": 1.0,
    },
    {
        "label": "Nucleus (temp=1.0, top_p=0.9)",
        "temperature": 1.0,
        "top_k": 0,
        "top_p": 0.9,
    },
    {
        "label": "Creative (temp=1.2, top_k=50, top_p=0.95)",
        "temperature": 1.2,
        "top_k": 50,
        "top_p": 0.95,
    },
]

comparison_rows = []
for i, strat in enumerate(SAMPLING_STRATEGIES):
    text = generate_text(
        COMPARE_PROMPT,
        max_new_tokens=280,
        temperature=strat["temperature"],
        top_k=strat["top_k"],
        top_p=strat["top_p"],
        seed=INFERENCE_CONFIG["seed"] + i,
    )
    comparison_rows.append({
        "strategy": strat["label"],
        "temperature": strat["temperature"],
        "top_k": strat["top_k"],
        "top_p": strat["top_p"],
        "chars": len(text),
        "preview": text[:200].replace("\n", " ") + ("..." if len(text) > 200 else ""),
    })
    print("=" * 72)
    print(strat["label"])
    print("=" * 72)
    print(text[:600])
    if len(text) > 600:
        print("...")
    print()

display(pd.DataFrame(comparison_rows))

Conservative (temp=0.5)
ROMEO:
The seat is the news to the duke is no more.

Nurse:
I am a with a pitiful and stronger,
The supply of the senators of the good and story,
The abbey of the earth the change of his death.

KING RICHARD II:
What have you the house of the king?

GLOUCESTER:
The king of his son the 

Balanced (temp=0.8, top_k=40)
ROMEO:
Why, then my life is fled by your good lordship.

MENENIUS:

DUKE VINCENTIO:
Methinks I have not the street?

MARCIUS:
Ay, the world of good Capitol Marcius: speak roars the hearts
With Marcius shame the possess.

VOLUMNIA:
There, then, then, the world that she say 'twixt dashin

Nucleus (temp=1.0, top_p=0.9)
ROMEO:
Give me leave my heart, in the leaves.

SICINIUS:
Marcius, home, go, sir, a very devilish one another
A gentleman's woman. Pray you, music have warrant
The people.

VOLUMNIA:
By your father well.

VOLUMNIA:
Hail, if you see here, he did lose the absence,
I thought must beheld my

Creative (temp=1.2, top_k=50, top_p=0.95)
ROMEO:
Thi

,strategy,temperature,top_k,top_p,chars,preview
0,Conservative (temp=0.5),0.5,0,1.00,286,ROMEO: The seat is the news to the duke is no ...
1,"Balanced (temp=0.8, top_k=40)",0.8,40,1.00,286,"ROMEO: Why, then my life is fled by your good ..."
2,"Nucleus (temp=1.0, top_p=0.9)",1.0,0,0.90,286,"ROMEO: Give me leave my heart, in the leaves. ..."
3,"Creative (temp=1.2, top_k=50, top_p=0.95)",1.2,50,0.95,286,ROMEO: This well-do you shall with teaches? a ...


## §8.2 Interactive Script Generator

Use the sliders to change the prompt, length, temperature, top-k, and top-p. Click **Generate** to sample a new continuation.

> Requires `ipywidgets`. In JupyterLab: `pip install ipywidgets` and enable the extension if needed.

In [9]:
if HAS_WIDGETS:
    prompt_widget = widgets.Textarea(
        value="ROMEO:\n",
        description="Prompt",
        layout=widgets.Layout(width="90%", height="80px"),
    )
    max_tokens_widget = widgets.IntSlider(
        value=INFERENCE_CONFIG["default_max_tokens"],
        min=50,
        max=800,
        step=50,
        description="Max tokens",
    )
    temp_widget = widgets.FloatSlider(
        value=0.8,
        min=0.1,
        max=2.0,
        step=0.05,
        description="Temperature",
        readout_format=".2f",
    )
    top_k_widget = widgets.IntSlider(
        value=40,
        min=0,
        max=65,
        step=1,
        description="Top-k (0=off)",
    )
    top_p_widget = widgets.FloatSlider(
        value=0.9,
        min=0.1,
        max=1.0,
        step=0.05,
        description="Top-p",
        readout_format=".2f",
    )
    seed_widget = widgets.IntText(value=INFERENCE_CONFIG["seed"], description="Seed")
    use_seed_widget = widgets.Checkbox(value=True, description="Fix seed")

    output_area = widgets.Output()

    def on_generate(_btn=None):
        with output_area:
            clear_output(wait=True)
            seed = seed_widget.value if use_seed_widget.value else None
            try:
                text = generate_text(
                    prompt_widget.value,
                    max_new_tokens=max_tokens_widget.value,
                    temperature=temp_widget.value,
                    top_k=top_k_widget.value,
                    top_p=top_p_widget.value,
                    seed=seed,
                )
                display(Markdown(f"```\n{text}\n```"))
            except Exception as exc:
                print(f"Error: {exc}")

    generate_btn = widgets.Button(description="Generate", button_style="primary")
    generate_btn.on_click(on_generate)

    controls = widgets.VBox([
        prompt_widget,
        widgets.HBox([max_tokens_widget, temp_widget]),
        widgets.HBox([top_k_widget, top_p_widget]),
        widgets.HBox([seed_widget, use_seed_widget, generate_btn]),
        output_area,
    ])
    display(controls)
    print("Adjust controls and click Generate.")
else:
    print("Install ipywidgets for the interactive UI: pip install ipywidgets")

Adjust controls and click Generate.


### Side-by-side A/B comparison

Generate two settings for the same prompt and view them next to each other.

In [10]:
if HAS_WIDGETS:
    ab_prompt = widgets.Text(value="KING:", description="Prompt")
    ab_tokens = widgets.IntSlider(value=300, min=50, max=600, step=25, description="Tokens")

    a_temp = widgets.FloatSlider(value=0.5, min=0.1, max=2.0, step=0.05, description="A temp")
    a_topk = widgets.IntSlider(value=0, min=0, max=65, description="A top-k")
    a_topp = widgets.FloatSlider(value=1.0, min=0.1, max=1.0, step=0.05, description="A top-p")

    b_temp = widgets.FloatSlider(value=1.0, min=0.1, max=2.0, step=0.05, description="B temp")
    b_topk = widgets.IntSlider(value=40, min=0, max=65, description="B top-k")
    b_topp = widgets.FloatSlider(value=0.9, min=0.1, max=1.0, step=0.05, description="B top-p")

    ab_out = widgets.Output()

    def compare_ab(_btn=None):
        with ab_out:
            clear_output(wait=True)
            text_a = generate_text(
                ab_prompt.value,
                max_new_tokens=ab_tokens.value,
                temperature=a_temp.value,
                top_k=a_topk.value,
                top_p=a_topp.value,
                seed=42,
            )
            text_b = generate_text(
                ab_prompt.value,
                max_new_tokens=ab_tokens.value,
                temperature=b_temp.value,
                top_k=b_topk.value,
                top_p=b_topp.value,
                seed=43,
            )
            col_a = widgets.HTML(f"<b>Setting A</b><pre style='white-space:pre-wrap'>{text_a}</pre>")
            col_b = widgets.HTML(f"<b>Setting B</b><pre style='white-space:pre-wrap'>{text_b}</pre>")
            display(widgets.HBox([col_a, col_b], layout=widgets.Layout(width="100%")))

    ab_btn = widgets.Button(description="Compare A vs B", button_style="info")
    ab_btn.on_click(compare_ab)

    display(widgets.VBox([
        ab_prompt,
        ab_tokens,
        widgets.HBox([a_temp, a_topk, a_topp]),
        widgets.HBox([b_temp, b_topk, b_topp]),
        ab_btn,
        ab_out,
    ]))
else:
    print("A/B comparison requires ipywidgets.")

## Optional: Log Demo Samples to W&B

In [11]:
LOG_TO_WANDB = False  # set True to log this demo run

if LOG_TO_WANDB and wandb is not None:
    run = wandb.init(
        project=INFERENCE_CONFIG["wandb_project"],
        entity=INFERENCE_CONFIG.get("wandb_entity"),
        job_type="inference-demo",
        name="inference-interactive-demo",
        config={**INFERENCE_CONFIG, "model_config": model_config},
    )
    table_rows = []
    for row in comparison_rows:
        full = generate_text(
            COMPARE_PROMPT,
            max_new_tokens=280,
            temperature=row["temperature"],
            top_k=int(row["top_k"]),
            top_p=row["top_p"],
            seed=42,
        )
        table_rows.append([row["strategy"], COMPARE_PROMPT, full[:3000]])
    wandb.log({
        "demo/generations": wandb.Table(
            columns=["strategy", "prompt", "text"],
            data=table_rows,
        )
    })
    run.finish()
    print("Logged demo generations to W&B")
else:
    print("W&B logging skipped (set LOG_TO_WANDB=True to enable)")

W&B logging skipped (set LOG_TO_WANDB=True to enable)


## Summary

| Proposal section | Implemented |
|------------------|-------------|
| §8.1 Inference | `generate_text(prompt, max_new_tokens)` + W&B artifact loader |
| §8.2 Interactive | Sliders for temperature, top-k, top-p + A/B comparison |

This completes the notebook pipeline: **explore → preprocess → baseline → sweep → train → evaluate → demo**.